In [8]:
# Option 3 with files and create metadata table to store file load details

from pyspark.sql.functions import *
from pyspark.sql.types import *




StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 10, Finished, Available, Finished, False)

In [24]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS processed_files
    (
        file_name STRING,
        load_time TIMESTAMP
    )
    USING DELTA
""")

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 26, Finished, Available, Finished, False)

DataFrame[]

In [25]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS Target_Sales_data
    (
        order_id INT,
        amount float,
        file_name STRING
    )
    USING DELTA
""")

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 27, Finished, Available, Finished, False)

DataFrame[]

In [26]:

df = spark.read.option("header",True).csv("Files/sales/")
df = df.withColumn("file_name",input_file_name())

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 28, Finished, Available, Finished, False)

In [27]:
display(df)

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c54dc1c4-1b91-43da-83b2-9339cb829846)

In [28]:
# Extract clean file name
df = df.withColumn(
    "file_name",regexp_extract("file_name",r'([^/]+$)',1)
)

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 30, Finished, Available, Finished, False)

In [29]:
display(df)

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 52532b94-69d3-46b5-b51f-23c66a7eadc3)

In [30]:
processed_df = spark.sql("SELECT file_name from processed_files")
display(processed_df)

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e0ccce56-ca6a-40cf-9b49-33216bcb7dee)

In [31]:
new_df = df.join(processed_df,on='file_name',how='left_anti')
display(new_df)

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c8a12e23-1cae-4e73-bee7-8ebc5440b229)

In [32]:

new_df = new_df.withColumn("order_id",col("order_id").cast("int"))\
.withColumn("amount",col("amount").cast("float"))

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 34, Finished, Available, Finished, False)

In [33]:
display(new_df)

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 67454927-65a4-4485-9880-5810e9d8cb2c)

In [34]:
new_df.write.format('delta').mode('append').saveAsTable("target_sales_data")

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 36, Finished, Available, Finished, False)

In [35]:
target_sales_data = spark.sql("select * from target_sales_data")
display(target_sales_data)

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0811b0c6-1b65-4a64-86c5-cb4c2f9d8cf6)

In [36]:
## existing file name load to processed_file
existing_files = (spark.table("target_sales_data")\
.select("file_name")\
.distinct()\
.withColumn("load_time",current_timestamp())
)
existing_files.write.format("delta")\
.mode("overwrite")\
.saveAsTable("processed_files")

StatementMeta(, 5b88f176-fce7-44dc-90d3-3120cddc3416, 38, Finished, Available, Finished, False)